# 26 — ICWS Weighted MinHash 512

Weighted MinHash / Consistent Weighted Sampling baseline for Weighted Jaccard. This produces a 512-sample sketch and ranks by sketch collision rate.

This is the most important non-neural Weighted Jaccard baseline for your paper.


In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np

sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import cleanup, eval_recall, load_dataset, save_result

# Edit here
dataset_name = "10k"       # ICWS brute sketch ranking is intended for 10k first
num_samples = 512
seed = 42
top_k = 500
OUT_PATH = "/tmp/results_sota_icws_512.pkl"
METHOD_NAME = "icws_weighted_minhash_512"
NOTEBOOK_NAME = "26_icws_weighted_minhash_512.ipynb"
np.random.seed(seed)


In [2]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
max_k = top_k


dataset=10k | qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)


In [3]:
# Ioffe-style Improved Consistent Weighted Sampling.
# Stores (dimension index, t) pairs; collision fraction estimates Weighted Jaccard.
def icws_signatures(x, num_samples=512, seed=42, batch_size=128):
    rng = np.random.default_rng(seed)
    d = x.shape[1]
    r = rng.gamma(shape=2.0, scale=1.0, size=(num_samples, d)).astype(np.float32)
    c = rng.gamma(shape=2.0, scale=1.0, size=(num_samples, d)).astype(np.float32)
    beta = rng.random(size=(num_samples, d), dtype=np.float32)

    sig_idx = np.empty((len(x), num_samples), dtype=np.int32)
    sig_t = np.empty((len(x), num_samples), dtype=np.int32)
    for start in range(0, len(x), batch_size):
        xb = np.maximum(x[start:start + batch_size], 1e-30).astype(np.float32)
        logx = np.log(xb)
        b = len(xb)
        idx_out = np.empty((b, num_samples), dtype=np.int32)
        t_out = np.empty((b, num_samples), dtype=np.int32)
        for s in range(num_samples):
            t = np.floor(logx / r[s] + beta[s]).astype(np.int32)
            y = np.exp(r[s] * (t - beta[s]))
            a = c[s] / (y * np.exp(r[s]))
            idx = np.argmin(a, axis=1)
            idx_out[:, s] = idx
            t_out[:, s] = t[np.arange(b), idx]
        sig_idx[start:start + b] = idx_out
        sig_t[start:start + b] = t_out
        print(f"signed {start + b:,}/{len(x):,}", flush=True)
    return sig_idx, sig_t

t0 = time.time()
sig_idx, sig_t = icws_signatures(qt, num_samples=num_samples, seed=seed)
print(f"signature time={(time.time()-t0)/60:.1f} min")
corpus_idx, query_idx = sig_idx[:query_start], sig_idx[query_start:]
corpus_t, query_t = sig_t[:query_start], sig_t[query_start:]
print(f"signature memory corpus={(corpus_idx.nbytes + corpus_t.nbytes)/1024**2:.1f} MB")


signed 128/10,000
signed 256/10,000
signed 384/10,000
signed 512/10,000
signed 640/10,000
signed 768/10,000
signed 896/10,000
signed 1,024/10,000
signed 1,152/10,000
signed 1,280/10,000
signed 1,408/10,000
signed 1,536/10,000
signed 1,664/10,000
signed 1,792/10,000
signed 1,920/10,000
signed 2,048/10,000
signed 2,176/10,000
signed 2,304/10,000
signed 2,432/10,000
signed 2,560/10,000
signed 2,688/10,000
signed 2,816/10,000
signed 2,944/10,000
signed 3,072/10,000
signed 3,200/10,000
signed 3,328/10,000
signed 3,456/10,000
signed 3,584/10,000
signed 3,712/10,000
signed 3,840/10,000
signed 3,968/10,000
signed 4,096/10,000
signed 4,224/10,000
signed 4,352/10,000
signed 4,480/10,000
signed 4,608/10,000
signed 4,736/10,000
signed 4,864/10,000
signed 4,992/10,000
signed 5,120/10,000
signed 5,248/10,000
signed 5,376/10,000
signed 5,504/10,000
signed 5,632/10,000
signed 5,760/10,000
signed 5,888/10,000
signed 6,016/10,000
signed 6,144/10,000
signed 6,272/10,000
signed 6,400/10,000
signed 6,528/1

In [4]:
def sketch_topk(query_idx, query_t, corpus_idx, corpus_t, k=500, batch_size=16):
    nbrs = []
    t0 = time.time()
    for start in range(0, len(query_idx), batch_size):
        qi = query_idx[start:start + batch_size]
        qt_ = query_t[start:start + batch_size]
        sim = ((qi[:, None, :] == corpus_idx[None, :, :]) &
               (qt_[:, None, :] == corpus_t[None, :, :])).mean(axis=2)
        kk = min(k, corpus_idx.shape[0])
        part = np.argpartition(-sim, kk - 1, axis=1)[:, :kk]
        rows = np.arange(len(qi))[:, None]
        order = np.argsort(-sim[rows, part], axis=1)
        top = part[rows, order]
        nbrs.extend([row.tolist() for row in top])
        print(f"ranked {start + len(qi):,}/{len(query_idx):,}", flush=True)
    qps = len(query_idx) / max(time.time() - t0, 1e-9)
    return nbrs, qps

nbrs, qps = sketch_topk(query_idx, query_t, corpus_idx, corpus_t, k=top_k)
metrics = {**eval_recall(gt, nbrs, query_start, top_k), "qps": qps, "samples": num_samples}
for k, v in metrics.items():
    if isinstance(k, int):
        print(f"R@{k:<4} = {v:.4f}")
print(f"QPS={qps:.1f}")
save_result(OUT_PATH, dataset_name, METHOD_NAME, metrics, meta={"notebook": NOTEBOOK_NAME})
cleanup()


ranked 16/2,000
ranked 32/2,000
ranked 48/2,000
ranked 64/2,000
ranked 80/2,000
ranked 96/2,000
ranked 112/2,000
ranked 128/2,000
ranked 144/2,000
ranked 160/2,000
ranked 176/2,000
ranked 192/2,000
ranked 208/2,000
ranked 224/2,000
ranked 240/2,000
ranked 256/2,000
ranked 272/2,000
ranked 288/2,000
ranked 304/2,000
ranked 320/2,000
ranked 336/2,000
ranked 352/2,000
ranked 368/2,000
ranked 384/2,000
ranked 400/2,000
ranked 416/2,000
ranked 432/2,000
ranked 448/2,000
ranked 464/2,000
ranked 480/2,000
ranked 496/2,000
ranked 512/2,000
ranked 528/2,000
ranked 544/2,000
ranked 560/2,000
ranked 576/2,000
ranked 592/2,000
ranked 608/2,000
ranked 624/2,000
ranked 640/2,000
ranked 656/2,000
ranked 672/2,000
ranked 688/2,000
ranked 704/2,000
ranked 720/2,000
ranked 736/2,000
ranked 752/2,000
ranked 768/2,000
ranked 784/2,000
ranked 800/2,000
ranked 816/2,000
ranked 832/2,000
ranked 848/2,000
ranked 864/2,000
ranked 880/2,000
ranked 896/2,000
ranked 912/2,000
ranked 928/2,000
ranked 944/2,000
ran